In [1]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path

# Heavy transformer imports are omitted to avoid protobuf incompatibility.
# The fallback model will always be used.
TRANSFORMERS_AVAILABLE = False

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.exceptions import NotFittedError



In [2]:
# Paths to data
DATA_ROOT = Path("/kaggle/input/learning-agency-lab-automated-essay-scoring-2")
train_path = DATA_ROOT / "train.csv"
train_df = pd.read_csv(train_path)

# Constant baseline (mean score) – used only if everything else fails
baseline_score = int(round(train_df["score"].mean()))
print(f"Baseline constant score: {baseline_score}")

# Train lightweight TF‑IDF + Ridge fallback model
print("Training lightweight TF‑IDF + Ridge fallback model...")
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english",
    dtype=np.float32,
)

train_texts = train_df["full_text"].astype(str).tolist()
X_train = vectorizer.fit_transform(train_texts)

y_train = train_df["score"].astype(float).values
# Use solver 'lsqr' which works with sparse matrices in current SciPy
ridge = Ridge(alpha=1.0, random_state=42, solver="lsqr")
ridge.fit(X_train, y_train)

# expose for later cells
simple_vectorizer = vectorizer
simple_model = ridge
print("Fallback model training completed.")



Baseline constant score: 3
Training lightweight TF‑IDF + Ridge fallback model...
Fallback model training completed.


In [3]:
test_path = DATA_ROOT / "test.csv"
test_df = pd.read_csv(test_path)

predicted_labels = None

if TRANSFORMERS_AVAILABLE:
    # This block will not run because transformers are unavailable.
    pass



In [4]:
# Use fallback model if transformer predictions are not available
if predicted_labels is None:
    try:
        X_test = simple_vectorizer.transform(test_df["full_text"].astype(str).tolist())
        preds = simple_model.predict(X_test)
        preds = np.rint(preds)  # round to nearest integer
        preds = np.clip(preds, 1, 6)  # enforce score bounds (1‑6)
        predicted_labels = preds.astype(int).tolist()
        print("Fallback model predictions generated.")
    except (NameError, NotFittedError) as e:
        print(f"Fallback model error ({e}), using constant baseline.")
        predicted_labels = [baseline_score] * len(test_df)

submission = pd.DataFrame(
    {
        "essay_id": test_df["essay_id"],
        "score": pd.Series(predicted_labels, dtype="int32"),
    }
)

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")



Fallback model predictions generated.
Submission written to submission.csv


In [5]:
# Quick look at the created submission
print(submission.head())

  essay_id  score
0  d550b2d      3
1  0c10954      2
2  ef04816      3
3  88ab8d9      2
4  cee8c0f      2
